# Step 6: SMH measurement mappings

St Mary's has no data dictionary. `smh_mappings.py` records which `interventionId` (and, for vitals, `interventionId_attributeId`; for medications, `materialId`) means which concept, taken from the extraction SQL comments and earlier analysis notebooks. The labels were not checked against a source dictionary, except for blood gas and labs, where `shortLabel` and the units in the tables can be used.

**What this notebook does**
1. Lists how many IDs are mapped per table.
2. Counts, for every SMH table in `derived/assembled/`, how many rows and stays each concept covers, how many rows are not mapped, and which IDs they are. The row count of every table is also compared with the count recorded at assembly, to show that the mapping check ran on the rebuilt tables.
3. Describes the largest unmapped IDs (value ranges for numbers, the commonest text for text) so that each is either mapped, deliberately left out, or listed as a question.
4. Saves the counts to `reports/`.

Output is counts and measurement codes only; no patient data.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import inventory as I, diagnostics as D, smh_mappings as SM

In [ ]:
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")  # keep whatever the assemble notebook uses
derived = open_picu_dir / "derived"
out_dir = derived / "assembled"
reports_dir = open_picu_dir / "reports"
assembled_baseline = pd.read_csv(reports_dir / "Assembled_counts_baseline.csv")
SM.to_table().groupby("table").size()

## 1. Coverage of the mapping in the rebuilt tables
For every SMH table, `SM.coverage_tables` maps each row's ID to its concept and reports:
- **Summary:** rows, rows mapped, % mapped, stays, and the number of unmapped IDs and their rows. The last two columns compare the table's row count with the count recorded when the tables were assembled (`Assembled_counts_baseline.csv`), so `same as assembly` should read True for every table.
- **Unmapped IDs:** the IDs with no concept, with their rows and stays.
- **Concepts, `<table>`:** rows and stays per concept.

An unmapped ID is not an error by itself. The questions are whether it is a measurement we need (then it gets a mapping), one we do not use (then it is recorded as left out), or one we can't identify (then it goes on the SMH question list).

In [ ]:
cov = SM.coverage_tables(out_dir, baseline=assembled_baseline)
cov["Summary"]

In [ ]:
cov["Unmapped IDs"]

In [ ]:
D.show({k: v for k, v in cov.items() if k.startswith("Concepts")})

## 2. What the unmapped IDs are
For the largest unmapped IDs, this shows how many rows and stays they have and what the values look like: numeric ranges (minimum, 5th and 95th percentile, median, maximum) or, for text, the most common values. Text values seen fewer than 10 times are hidden. The IDs below are the ones found in the first pass; replace them with the IDs the table above lists if they differ.

In [ ]:
SM.describe_ids(out_dir, "vitals", ["3406_57907"], "terseForm")
SM.describe_ids(out_dir, "infusions", ["11858", "13245", "11999", "10706"], "Drug_terse")

## 3. Save
The coverage tables are written to `reports/`. They hold counts and measurement codes only.

In [ ]:
import contextlib, io

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    SM.describe_ids(out_dir, "vitals", ["3406_57907"], "terseForm")
    SM.describe_ids(out_dir, "infusions", ["11858", "13245", "11999", "10706"], "Drug_terse")

path = reports_dir / f"SMH_mappings_{pd.Timestamp.today().date()}.md"
D.write_report({"Summary": cov["Summary"], "Unmapped IDs": cov["Unmapped IDs"]}, path,
               title="SMH measurement mappings")
with open(path, "a") as f:
    f.write("\n## Values of the largest unmapped IDs\n\n```\n" + buf.getvalue() + "```\n")
path